# Unit 3 Assignment: Building a Production Advanced RAG System

**Topic:** Advanced RAG - Retrieval Enhancement, Re-Ranking, and Query Expansion

This notebook implements a full Advanced RAG pipeline combining:
- Hybrid Retrieval (BM25 + SBERT + RRF)
- Cross-Encoder Re-Ranking
- Query Expansion (HyDE via Gemini)
- End-to-End Pipeline with LLM Generation (Groq)

In [2]:
!pip install rank-bm25 sentence-transformers google-generativeai groq numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.9 MB/s eta 0:00:00


In [5]:
import os
import numpy as np
from typing import List, Dict

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
from sentence_transformers.cross_encoder import CrossEncoder
import google.generativeai as genai
from groq import Groq

import os
import getpass

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter Google API Key: ")

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Groq API Key: ")

print(f"GOOGLE key loaded: {'yes' if os.getenv('GOOGLE_API_KEY') else 'NO'}")
print(f"GROQ   key loaded: {'yes' if os.getenv('GROQ_API_KEY') else 'NO'}")

Enter Google API Key: ··········
Enter Groq API Key: ··········
GOOGLE key loaded: yes
GROQ   key loaded: yes


## Part 1 - Document Corpus Setup

In [6]:
CORPUS = [
    # Attention / Transformers (3 related docs)
    "The attention mechanism allows a model to weigh the importance of different tokens "
    "when encoding a sequence, enabling it to capture long-range dependencies.",

    "Scaled dot-product attention computes compatibility between queries and keys using a "
    "dot product scaled by the square root of the key dimension, then applies a softmax "
    "to obtain attention weights over values.",

    "Multi-head attention runs several attention functions in parallel, each projecting "
    "queries, keys, and values into a lower-dimensional subspace, and concatenates the "
    "results to form the final representation.",

    # Neural Network Training (3 related docs)
    "Stochastic gradient descent (SGD) updates model parameters by computing the gradient "
    "of the loss on a small mini-batch, making training tractable for large datasets.",

    "Adam optimiser combines momentum and adaptive learning rates: it maintains exponential "
    "moving averages of both the gradient and the squared gradient to adaptively scale "
    "each parameter update step.",

    "Batch normalisation standardises layer inputs to have zero mean and unit variance "
    "during training, which accelerates convergence and acts as a regulariser, reducing "
    "the need for dropout.",

    # Embeddings and Representations
    "Word embeddings such as Word2Vec and GloVe map discrete tokens to dense vectors "
    "in a continuous space where semantic similarity corresponds to geometric proximity.",

    "Positional encodings are added to token embeddings in transformer models to inject "
    "information about the order of tokens, since self-attention is permutation-invariant.",

    # Regularisation
    "Dropout randomly zeroes a fraction of neuron activations during training, forcing "
    "the network to learn redundant representations and thereby reducing overfitting.",

    # Technical jargon doc (BM25-friendly)
    "The BERT (Bidirectional Encoder Representations from Transformers) pre-training "
    "objective uses Masked Language Modelling (MLM) and Next Sentence Prediction (NSP) "
    "to learn deep bidirectional representations from unlabelled text.",

    # Extra docs
    "Convolutional neural networks (CNNs) use learnable filters that slide over the input "
    "to extract local spatial features, making them highly effective for image recognition "
    "tasks thanks to their weight-sharing property.",

    "Recurrent neural networks (RNNs) process sequential data by maintaining a hidden "
    "state that is updated at each timestep, but they struggle with long-range dependencies "
    "due to the vanishing gradient problem.",

    "The vanishing gradient problem occurs when gradients shrink exponentially as they "
    "propagate back through many layers, making it difficult to train deep networks with "
    "saturating activations like sigmoid or tanh.",

    "Transfer learning fine-tunes a model pre-trained on a large dataset on a smaller "
    "task-specific dataset, achieving strong performance even with limited labelled data.",
]

print(f'Corpus size: {len(CORPUS)} documents')
for i, doc in enumerate(CORPUS):
    print(f'  [{i:02d}] {doc[:80]}...')

Corpus size: 14 documents
  [00] The attention mechanism allows a model to weigh the importance of different toke...
  [01] Scaled dot-product attention computes compatibility between queries and keys usi...
  [02] Multi-head attention runs several attention functions in parallel, each projecti...
  [03] Stochastic gradient descent (SGD) updates model parameters by computing the grad...
  [04] Adam optimiser combines momentum and adaptive learning rates: it maintains expon...
  [05] Batch normalisation standardises layer inputs to have zero mean and unit varianc...
  [06] Word embeddings such as Word2Vec and GloVe map discrete tokens to dense vectors ...
  [07] Positional encodings are added to token embeddings in transformer models to inje...
  [08] Dropout randomly zeroes a fraction of neuron activations during training, forcin...
  [09] The BERT (Bidirectional Encoder Representations from Transformers) pre-training ...
  [10] Convolutional neural networks (CNNs) use learnable filter

## Part 2 - HybridRetriever (BM25 + SBERT + RRF)

In [7]:
class HybridRetriever:
    """
    Combines BM25 (keyword) and SBERT (dense semantic) retrieval
    using Reciprocal Rank Fusion (RRF) to produce a single ranked list.
    """

    def __init__(self, corpus: List[str], k: int = 60):
        self.corpus = corpus
        self.k = k

        # BM25 index - tokenise on whitespace after lowercasing
        tokenised = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenised)
        print('BM25 index built.')

        # SBERT index
        self.sbert = SentenceTransformer('all-MiniLM-L6-v2')
        self.doc_embeddings = self.sbert.encode(
            corpus, convert_to_tensor=True, show_progress_bar=True
        )
        print('SBERT embeddings computed.')

    def _bm25_ranks(self, query: str) -> Dict[int, int]:
        """Return {doc_id: rank} ranked by BM25 score (1 = best)."""
        scores = self.bm25.get_scores(query.lower().split())
        ranked_ids = np.argsort(scores)[::-1]
        return {int(doc_id): rank + 1 for rank, doc_id in enumerate(ranked_ids)}

    def _sbert_ranks(self, query: str) -> Dict[int, int]:
        """Return {doc_id: rank} ranked by cosine similarity (1 = best)."""
        query_emb = self.sbert.encode(query, convert_to_tensor=True)
        cos_scores = util.cos_sim(query_emb, self.doc_embeddings)[0].cpu().numpy()
        ranked_ids = np.argsort(cos_scores)[::-1]
        return {int(doc_id): rank + 1 for rank, doc_id in enumerate(ranked_ids)}

    def _rrf_score(self, rank: int) -> float:
        """Reciprocal Rank Fusion score for a single rank."""
        return 1.0 / (self.k + rank)

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        """
        Retrieve top_k documents using RRF over BM25 and SBERT.
        Returns list of dicts with keys:
            doc_id, rrf_score, bm25_rank, sbert_rank, text
        """
        bm25_ranks  = self._bm25_ranks(query)
        sbert_ranks = self._sbert_ranks(query)

        results = []
        for doc_id in range(len(self.corpus)):
            r_bm25  = bm25_ranks[doc_id]
            r_sbert = sbert_ranks[doc_id]
            rrf     = self._rrf_score(r_bm25) + self._rrf_score(r_sbert)
            results.append({
                'doc_id'    : doc_id,
                'rrf_score' : rrf,
                'bm25_rank' : r_bm25,
                'sbert_rank': r_sbert,
                'text'      : self.corpus[doc_id],
            })

        results.sort(key=lambda x: x['rrf_score'], reverse=True)
        return results[:top_k]


retriever = HybridRetriever(CORPUS, k=60)
print('HybridRetriever ready.')

# Sanity check
test_results = retriever.retrieve('how does attention work?', top_k=3)
print('\nQuery: how does attention work?')
for r in test_results:
    print(f"  doc_id={r['doc_id']}  rrf={r['rrf_score']:.5f}  "
          f"bm25_rank={r['bm25_rank']}  sbert_rank={r['sbert_rank']}")
    print(f"  -> {r['text'][:90]}...")

BM25 index built.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

SBERT embeddings computed.
HybridRetriever ready.

Query: how does attention work?
  doc_id=2  rrf=0.03252  bm25_rank=1  sbert_rank=2
  -> Multi-head attention runs several attention functions in parallel, each projecting queries...
  doc_id=0  rrf=0.03227  bm25_rank=3  sbert_rank=1
  -> The attention mechanism allows a model to weigh the importance of different tokens when en...
  doc_id=1  rrf=0.03200  bm25_rank=2  sbert_rank=3
  -> Scaled dot-product attention computes compatibility between queries and keys using a dot p...


## Part 3 - Cross-Encoder Re-Ranker

In [8]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('Cross-encoder loaded.')


def rerank(query: str, candidates: List[Dict], top_k: int = 3) -> List[Dict]:
    """
    Re-rank candidate documents using the cross-encoder.

    Parameters
    ----------
    query      : ORIGINAL user query (not the HyDE-expanded version)
    candidates : output of HybridRetriever.retrieve()
    top_k      : number of documents to return after re-ranking

    Returns list of dicts enriched with 'ce_score', sorted descending.
    Note: cross-encoder scores can be negative - that is normal.
    """
    pairs = [(query, c['text']) for c in candidates]
    ce_scores = cross_encoder.predict(pairs)

    for cand, score in zip(candidates, ce_scores):
        cand['ce_score'] = float(score)

    reranked = sorted(candidates, key=lambda x: x['ce_score'], reverse=True)
    return reranked[:top_k]


# Quick test
reranked = rerank('how does attention work?', test_results, top_k=3)
print('Re-ranked results:')
for r in reranked:
    print(f"  ce_score={r['ce_score']:.4f}  -> {r['text'][:80]}...")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Cross-encoder loaded.
Re-ranked results:
  ce_score=6.0954  -> The attention mechanism allows a model to weigh the importance of different toke...
  ce_score=3.2549  -> Multi-head attention runs several attention functions in parallel, each projecti...
  ce_score=1.1691  -> Scaled dot-product attention computes compatibility between queries and keys usi...


## Part 4 - Query Expansion via HyDE



In [10]:
def hyde_expand(user_query: str) -> str:
    """
    Hypothetical Document Embedding (HyDE) query expansion.
    Prompts Gemini to write a short factual answer using precise
    technical vocabulary. temperature=0.0 for deterministic output.
    """
    prompt = (
        'You are an AI/ML textbook author. '
        'Write a concise 2-3 sentence factual answer to the following question. '
        'Use precise technical vocabulary.\n\n'
        f'Question: {user_query}\n\n'
        'Answer:'
    )

    model = genai.GenerativeModel(
        model_name='gemini-2.5-flash',
        generation_config=genai.types.GenerationConfig(temperature=0.0),
    )
    response = model.generate_content(prompt)
    return response.text.strip()



sample_query = 'what is attention?'
expanded = hyde_expand(sample_query)
print(f'Original query : {sample_query}')
print(f'HyDE expansion : {expanded}')

Original query : what is attention?
HyDE expansion : Attention is a mechanism in neural networks that allows the model to selectively focus on different parts of its input sequence when processing information or generating an output. It computes a set of importance weights for each input element, indicating its relevance to the current task or target element. These weights are then used to create a weighted sum of the input elements, forming a context vector that captures the most pertinent information.


## Part 5 - End-to-End Advanced RAG Pipeline

In [11]:
def generate_answer(user_query: str, context_docs: List[Dict]) -> str:
    """Call Groq LLM to produce a final answer given the user query and context."""
    context = '\n\n'.join(
        [f'[Doc {i+1}] {d["text"]}' for i, d in enumerate(context_docs)]
    )
    system_prompt = (
        'You are a helpful university AI/ML teaching assistant. '
        'Answer the student question using ONLY the provided context. '
        'Be concise and precise.'
    )
    user_prompt = (
        f'Context:\n{context}\n\n'
        f'Student Question: {user_query}\n\n'
        'Answer:'
    )

    response = groq_client.chat.completions.create(
        model='llama3-8b-8192',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt},
        ],
        temperature=0.2,
        max_tokens=300,
    )
    return response.choices[0].message.content.strip()


def advanced_rag(
    user_query: str,
    retrieval_top_k: int = 5,
    rerank_top_k: int = 3,
    verbose: bool = True,
) -> str:
    """
    Full pipeline:
        1. Query Expansion  (HyDE via Gemini)
        2. Hybrid Retrieval (BM25 + SBERT + RRF) using expanded query
        3. Cross-Encoder Re-Ranking using the ORIGINAL user query
        4. LLM Generation  (Groq / LLaMA-3)

    Returns the final answer string.
    """
    if verbose:
        print('=' * 60)
        print(f'USER QUERY: {user_query}')
        print('=' * 60)

    # Step 1: HyDE Query Expansion
    expanded_query = hyde_expand(user_query)
    if verbose:
        print(f'\n[1] HyDE Expansion:\n{expanded_query}\n')

    # Step 2: Hybrid Retrieval on expanded query
    candidates = retriever.retrieve(expanded_query, top_k=retrieval_top_k)
    if verbose:
        print(f'[2] Hybrid Retrieval (top {retrieval_top_k}):')
        for c in candidates:
            print(f"    rrf={c['rrf_score']:.5f}  bm25_rank={c['bm25_rank']}  "
                  f"sbert_rank={c['sbert_rank']}  -> {c['text'][:65]}...")

    # Step 3: Cross-Encoder Re-Ranking on ORIGINAL query
    top_docs = rerank(user_query, candidates, top_k=rerank_top_k)
    if verbose:
        print(f'\n[3] After Re-Ranking (top {rerank_top_k}):')
        for d in top_docs:
            print(f"    ce_score={d['ce_score']:.4f}  -> {d['text'][:65]}...")

    # Step 4: LLM Generation
    answer = generate_answer(user_query, top_docs)
    if verbose:
        print(f'\n[4] FINAL ANSWER:\n{answer}')
        print('=' * 60)

    return answer


print('advanced_rag() pipeline defined.')

advanced_rag() pipeline defined.


## Part 6 - Comparison Experiment

Naive RAG = dense-only SBERT retrieval, no expansion, no re-ranking.
Advanced RAG = full pipeline from Part 5.

In [13]:
def naive_rag(user_query: str) -> Dict:
    """Naive RAG: SBERT cosine only, no expansion, no re-ranking."""
    query_emb = retriever.sbert.encode(user_query, convert_to_tensor=True)
    cos_scores = util.cos_sim(query_emb, retriever.doc_embeddings)[0].cpu().numpy()
    top_idx = int(np.argsort(cos_scores)[::-1][0])
    return {
        'doc_id'      : top_idx,
        'sbert_score' : float(cos_scores[top_idx]),
        'text'        : CORPUS[top_idx],
    }


def advanced_rag_top_doc(user_query: str) -> Dict:
    """Run full advanced pipeline, return only the top re-ranked document."""
    expanded_query = hyde_expand(user_query)
    candidates     = retriever.retrieve(expanded_query, top_k=5)
    top_docs       = rerank(user_query, candidates, top_k=1)
    return top_docs[0]


TEST_QUERIES = [
    'how do transformers encode meaning?',
    'optimization techniques for training',
    'why do deep networks suffer from vanishing gradients?',
]

comparison_rows = []

for query in TEST_QUERIES:
    print(f"\nRunning: '{query}'")
    naive_top = naive_rag(query)
    adv_top   = advanced_rag_top_doc(query)
    same      = naive_top['doc_id'] == adv_top['doc_id']
    comparison_rows.append({
        'query'        : query,
        'naive_doc_id' : naive_top['doc_id'],
        'naive_text'   : naive_top['text'][:80],
        'adv_doc_id'   : adv_top['doc_id'],
        'adv_text'     : adv_top['text'][:80],
        'different'    : 'No' if same else 'Yes',
    })
    print(f"  Naive  [id={naive_top['doc_id']}]: {naive_top['text'][:70]}...")
    print(f"  Adv    [id={adv_top['doc_id']}]:   {adv_top['text'][:70]}...")
    print(f"  Different? {comparison_rows[-1]['different']}")

print('\nComparison complete.')


Running: 'how do transformers encode meaning?'
  Naive  [id=7]: Positional encodings are added to token embeddings in transformer mode...
  Adv    [id=9]:   The BERT (Bidirectional Encoder Representations from Transformers) pre...
  Different? Yes

Running: 'optimization techniques for training'
  Naive  [id=3]: Stochastic gradient descent (SGD) updates model parameters by computin...
  Adv    [id=5]:   Batch normalisation standardises layer inputs to have zero mean and un...
  Different? Yes

Running: 'why do deep networks suffer from vanishing gradients?'
  Naive  [id=12]: The vanishing gradient problem occurs when gradients shrink exponentia...
  Adv    [id=12]:   The vanishing gradient problem occurs when gradients shrink exponentia...
  Different? No

Comparison complete.


In [14]:
# Printing the summary table from above to compare
header = f"{'Query':<50} | {'Naive top doc id':<17} | {'Adv top doc id':<15} | Different?"
print(header)
print('-' * len(header))
for row in comparison_rows:
    print(f"{row['query']:<50} | {row['naive_doc_id']:<17} | {row['adv_doc_id']:<15} | {row['different']}")

Query                                              | Naive top doc id  | Adv top doc id  | Different?
-----------------------------------------------------------------------------------------------------
how do transformers encode meaning?                | 7                 | 9               | Yes
optimization techniques for training               | 3                 | 5               | Yes
why do deep networks suffer from vanishing gradients? | 12                | 12              | No


---
## Bonus 1 - Weighted RRF


In [15]:
def weighted_rrf_retrieve(
    retr: HybridRetriever,
    query: str,
    alpha: float = 0.5,
    top_k: int = 5,
) -> List[Dict]:
    """
    Weighted RRF retrieval.
    alpha=0.5 is standard RRF; adjust to favour BM25 or SBERT.
    """
    bm25_ranks  = retr._bm25_ranks(query)
    sbert_ranks = retr._sbert_ranks(query)

    results = []
    for doc_id in range(len(retr.corpus)):
        rrf = (
            alpha       * retr._rrf_score(bm25_ranks[doc_id])
            + (1-alpha) * retr._rrf_score(sbert_ranks[doc_id])
        )
        results.append({
            'doc_id'    : doc_id,
            'rrf_score' : rrf,
            'bm25_rank' : bm25_ranks[doc_id],
            'sbert_rank': sbert_ranks[doc_id],
            'text'      : retr.corpus[doc_id],
            'alpha'     : alpha,
        })

    results.sort(key=lambda x: x['rrf_score'], reverse=True)
    return results[:top_k]


keyword_query  = 'BERT MLM NSP pre-training'
semantic_query = 'how do models understand context in a sentence?'

print('-- Keyword-heavy query --')
for alpha in [0.3, 0.5, 0.7]:
    top = weighted_rrf_retrieve(retriever, keyword_query, alpha=alpha, top_k=1)[0]
    print(f"  alpha={alpha}  top_doc=[{top['doc_id']}]  -> {top['text'][:70]}...")

print('\n-- Semantic query --')
for alpha in [0.3, 0.5, 0.7]:
    top = weighted_rrf_retrieve(retriever, semantic_query, alpha=alpha, top_k=1)[0]
    print(f"  alpha={alpha}  top_doc=[{top['doc_id']}]  -> {top['text'][:70]}...")

-- Keyword-heavy query --
  alpha=0.3  top_doc=[9]  -> The BERT (Bidirectional Encoder Representations from Transformers) pre...
  alpha=0.5  top_doc=[9]  -> The BERT (Bidirectional Encoder Representations from Transformers) pre...
  alpha=0.7  top_doc=[9]  -> The BERT (Bidirectional Encoder Representations from Transformers) pre...

-- Semantic query --
  alpha=0.3  top_doc=[7]  -> Positional encodings are added to token embeddings in transformer mode...
  alpha=0.5  top_doc=[7]  -> Positional encodings are added to token embeddings in transformer mode...
  alpha=0.7  top_doc=[7]  -> Positional encodings are added to token embeddings in transformer mode...


---
## Bonus 2 - Chunk Size Study

In [16]:
LONG_DOC = (
    'Transformers have become the dominant architecture in natural language processing. '
    'The core innovation is the self-attention mechanism, which allows each token to attend '
    'to every other token simultaneously, unlike recurrent networks that process tokens '
    'sequentially and struggle with long-range dependencies due to the vanishing gradient problem. '
    'Attention computes a weighted sum of value vectors using softmax-normalised dot products '
    'between queries and keys, scaled by sqrt(d_k) to prevent large magnitudes. '
    'Multi-head attention runs h heads in parallel, each attending to different subspaces, '
    'then concatenates and projects their outputs. '
    'Positional encodings are added to embeddings because self-attention is permutation-invariant. '
    'Feed-forward sub-layers, residual connections, and layer normalisation complete each block. '
    'Training uses Adam with a warm-up schedule and dropout for regularisation. '
    'Large models like BERT, GPT, and T5 show that pre-training on large corpora '
    'followed by fine-tuning achieves state-of-the-art performance across NLP tasks. '
    'The BERT pre-training objective uses Masked Language Modelling and Next Sentence Prediction '
    'to learn deep bidirectional representations from unlabelled text. '
    'GPT uses a causal language modelling objective, predicting each token from prior context only. '
    'T5 frames all NLP tasks as text-to-text problems, unifying pre-training and fine-tuning. '
    'Scaling laws suggest that model performance improves predictably with more parameters and data. '
    'Instruction tuning and reinforcement learning from human feedback further align large models '
    'with human preferences and task requirements.'
)


def chunk_text(text: str, chunk_size: int) -> List[str]:
    words = text.split()
    return [' '.join(words[i: i + chunk_size]) for i in range(0, len(words), chunk_size)]


study_query = 'how does multi-head attention work?'
print(f'Query: {study_query}\n')

for chunk_words in [50, 100, 200]:
    chunks   = chunk_text(LONG_DOC, chunk_words)
    tmp_retr = HybridRetriever(chunks, k=60)
    top      = tmp_retr.retrieve(study_query, top_k=1)[0]
    print(f'chunk_size={chunk_words} words -> {len(chunks)} chunks')
    print(f'  Top chunk: {top["text"][:110]}...')
    print(f'  RRF score: {top["rrf_score"]:.5f}\n')

Query: how does multi-head attention work?

BM25 index built.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

SBERT embeddings computed.
chunk_size=50 words -> 5 chunks
  Top chunk: sum of value vectors using softmax-normalised dot products between queries and keys, scaled by sqrt(d_k) to pr...
  RRF score: 0.03279

BM25 index built.


...

SBERT embeddings computed.
chunk_size=100 words -> 3 chunks
  Top chunk: Transformers have become the dominant architecture in natural language processing. The core innovation is the ...
  RRF score: 0.03279

BM25 index built.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

SBERT embeddings computed.
chunk_size=200 words -> 2 chunks
  Top chunk: Transformers have become the dominant architecture in natural language processing. The core innovation is the ...
  RRF score: 0.03252



In [17]:
# ── Bonus 3: ColBERT MaxSim as a third retriever ──────────────────────────────

def colbert_maxsim_scores(query: str, corpus: list, model: SentenceTransformer) -> np.ndarray:
    """
    ColBERT-style MaxSim scoring.

    For each document, compute token-level embeddings for both the query
    and the document, then score = sum of max cosine similarities between
    each query token and all document tokens (MaxSim).
    """
    # Encode query tokens individually (each word as a separate 'sentence')
    query_tokens = query.lower().split()
    doc_tokens_list = [doc.lower().split() for doc in corpus]

    # Get embeddings for each query token
    query_token_embs = model.encode(query_tokens, convert_to_tensor=True)  # (Q, D)

    scores = []
    for doc_tokens in doc_tokens_list:
        # Get embeddings for each document token
        doc_token_embs = model.encode(doc_tokens, convert_to_tensor=True)  # (T, D)

        # Cosine similarity matrix: (Q x T)
        sim_matrix = util.cos_sim(query_token_embs, doc_token_embs)  # (Q, T)

        # MaxSim: for each query token, take max similarity over all doc tokens
        max_sims = sim_matrix.max(dim=1).values  # (Q,)

        # Final score = sum of MaxSim values
        score = max_sims.sum().item()
        scores.append(score)

    return np.array(scores)


def colbert_ranks(query: str, corpus: list, model: SentenceTransformer) -> Dict[int, int]:
    """Return {doc_id: rank} ranked by ColBERT MaxSim score (1 = best)."""
    scores = colbert_maxsim_scores(query, corpus, model)
    ranked_ids = np.argsort(scores)[::-1]
    return {int(doc_id): rank + 1 for rank, doc_id in enumerate(ranked_ids)}


def three_way_rrf_retrieve(
    retr: HybridRetriever,
    query: str,
    top_k: int = 5,
    k: int = 60,
) -> List[Dict]:
    """
    Three-way RRF fusion: BM25 + SBERT + ColBERT.

    score(d) = 1/(k + r_BM25(d)) + 1/(k + r_SBERT(d)) + 1/(k + r_ColBERT(d))
    """
    bm25_r   = retr._bm25_ranks(query)
    sbert_r  = retr._sbert_ranks(query)
    colbert_r = colbert_ranks(query, retr.corpus, retr.sbert)  # reuse SBERT model

    results = []
    for doc_id in range(len(retr.corpus)):
        rrf = (
            1.0 / (k + bm25_r[doc_id])
            + 1.0 / (k + sbert_r[doc_id])
            + 1.0 / (k + colbert_r[doc_id])
        )
        results.append({
            'doc_id'       : doc_id,
            'rrf_score'    : rrf,
            'bm25_rank'    : bm25_r[doc_id],
            'sbert_rank'   : sbert_r[doc_id],
            'colbert_rank' : colbert_r[doc_id],
            'text'         : retr.corpus[doc_id],
        })

    results.sort(key=lambda x: x['rrf_score'], reverse=True)
    return results[:top_k]


# ── Compare 2-way RRF vs 3-way RRF ───────────────────────────────────────────
test_queries = [
    'how do transformers encode meaning?',
    'optimization techniques for training',
    'why do deep networks suffer from vanishing gradients?',
]

print(f"{'Query':<50} | {'2-way top doc':<35} | {'3-way top doc':<35} | Different?")
print('-' * 135)

for q in test_queries:
    two_way   = retriever.retrieve(q, top_k=1)[0]
    three_way = three_way_rrf_retrieve(retriever, q, top_k=1)[0]
    different = 'Yes' if two_way['doc_id'] != three_way['doc_id'] else 'No'
    print(f"{q:<50} | [{two_way['doc_id']}] {two_way['text'][:30]:<32}| [{three_way['doc_id']}] {three_way['text'][:30]:<32}| {different}")

Query                                              | 2-way top doc                       | 3-way top doc                       | Different?
---------------------------------------------------------------------------------------------------------------------------------------
how do transformers encode meaning?                | [9] The BERT (Bidirectional Encode  | [9] The BERT (Bidirectional Encode  | No
optimization techniques for training               | [3] Stochastic gradient descent (S  | [3] Stochastic gradient descent (S  | No
why do deep networks suffer from vanishing gradients? | [12] The vanishing gradient problem  | [12] The vanishing gradient problem  | No
